# 15-amaliyot — Hujjat yordamchisi agenti (m15) · KAPSTONE YAKUNI

**Mavzu:** LangChain ReAct agenti — butun kurs davomida qurilgan modullarni bitta agentga birlashtiramiz.
**Kuni:** 16-kun (Day 16) · **Juft ma'ruza:** [L15 — AI agentlar](../lectures/d15_agentlar.pdf)
**Quriladigan kapstone moduli:** `m15 DocumentAssistantAgent` (yakuniy modul — himoya demo'sida ishlatiladi).

Bu **kapstone cho'qqisi**: m13 (sentiment), m14 (RAG), m12 (xulosa), m04 (imlo), m10 (entity) — barchasi
agent **vositalariga** aylanadi. Agent so'rovga qarab to'g'ri vositani tanlaydi (L15 ReAct).

**Bugungi maqsadlar (L15 [C] dan):**
1. ReAct tool tanlashni tekshiramiz (L15 [I1]: \texttt{sentiment\_classify}, ishonch $>0.7$).
2. 5 ta modulni `Tool` sifatida agentga ulaymiz.
3. Murakkab so'rovni agent orqali bajarib, Thought→Action→Observation izini kuzatamiz.

> ⚠️ **session_role:** wiring_and_polish — bu seans mavjud modullarni ulash va sayqallash (noldan emas).
> **Offline:** langchain/LLM o'rniga o'zbekcha kalit so'zli **rule-based router** ishlaydi.

## 1. Muhit tekshiruvi

Kutubxonalar, urug' va bayroqlar. `langchain`/LLM API bo'lmasa ham notebook uchdan-uchgacha ishlaydi:
offline rejimda **rule-based keyword router** (ReAct "Thought" ni modellaydi) ishlatiladi.

In [ ]:
import os, sys, random, warnings
warnings.filterwarnings("ignore")
import numpy as np

OFFLINE_FALLBACK = True          # mahalliy: langchain/LLM yo'q -> rule-based router
random.seed(42); np.random.seed(42)

try:
    import langchain              # noqa: F401
    HAS_LANGCHAIN = True
except Exception:
    HAS_LANGCHAIN = False

USE_LANGCHAIN = HAS_LANGCHAIN and not OFFLINE_FALLBACK
USE_LLM = False                  # LLM API kaliti + internet (Kaggle)

print("langchain:", ("bor" if HAS_LANGCHAIN else "yo'q"),
      "| USE_LANGCHAIN:", USE_LANGCHAIN, "| USE_LLM:", USE_LLM)

for _cand in ["../../capstone/modules", "/kaggle/working/capstone/modules", "capstone/modules"]:
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand); break

## 2. Yaxlit natija (avval manzil)

Tugallangan natijani ko'ramiz: agentga so'rov beramiz, u to'g'ri vositani tanlab javob qaytaradi va
ReAct izini (Thought→Action→Observation) ko'rsatadi.

In [ ]:
import m15_langchain_agent as M15
from m15_langchain_agent import DocumentAssistantAgent
M15.USE_LANGCHAIN = USE_LANGCHAIN     # offline -> False (rule-based router)

# Kichik offline data (tool'lar uchun)
SENT_TEXTS = ["mahsulot zo'r", "juda yoqdi", "ajoyib xizmat",
              "mahsulot yomon", "umuman yoqmadi", "sifati past"]
SENT_LABELS = ["ijobiy", "ijobiy", "ijobiy", "salbiy", "salbiy", "salbiy"]
KB = ["Toshkent O'zbekiston poytaxti.",
      "Mehnat shartnomasi xodim va ish beruvchi o'rtasida tuziladi.",
      "Soliq to'lash fuqaroning majburiyati."]

def make_tools():
    """5 kapstone modulini tool callable'ga ulaydi (mavjud bo'lmasa zaxira)."""
    tools = {}
    try:
        import m13_bert_classifier as _m13; _m13.USE_TRANSFORMERS = False
        clf = _m13.FineTunedClassifier(); clf.fit(SENT_TEXTS, SENT_LABELS)
        tools["sentiment_classify"] = clf.predict
    except Exception:
        tools["sentiment_classify"] = lambda t: "ijobiy"
    try:
        import m14_rag_engine as _m14; _m14.USE_ST = False; _m14.USE_LLM = False
        rag = _m14.RAGEngine(); rag.index(KB)
        tools["retrieve_docs"] = lambda q: rag.answer(q, k=2)["answer"]
    except Exception:
        tools["retrieve_docs"] = lambda q: "(qidiruv mavjud emas)"
    try:
        import m12_transformer_summarizer as _m12; _m12.HAS_TORCH = False
        summ = _m12.TransformerSummarizer()
        tools["summarize_text"] = lambda t: summ.summarize(t, max_length=20)
    except Exception:
        tools["summarize_text"] = lambda t: t[:60]
    tools["spell_correct"] = lambda w: w               # soddalashtirilgan
    tools["extract_entities"] = lambda t: []           # soddalashtirilgan
    return tools

agent = DocumentAssistantAgent(tools=make_tools())
print(agent.run("Uzum Market ilovasi yaxshimi?"))
print("Iz:", agent.last_trace[0]["action"], "->", agent.last_trace[1]["observation"])

## 3. Tayyor kod bloki — periferiya (PRIMM)

Kaggle yo'lida agent **LangChain** bilan quriladi (LLM rejalashtiradi). Offline rejimda esa o'zbekcha
**kalit so'zli router** ReAct "Thought" ni modellaydi.

### Bashorat qiling
Agentga "Bu hujjatni qisqacha bayon qil" desangiz, qaysi vosita tanlanadi? Kalit so'z qaysi?

In [ ]:
# Kaggle yo'li (USE_LANGCHAIN=True, LLM) -- ko'rsatiladi, mahalliyda bajarilmaydi:
#   from langchain.agents import Tool, AgentExecutor, create_react_agent
#   from langchain import hub
#   lc_tools = [Tool(name=n, func=f, description=n) for n, f in tools.items()]
#   agent = create_react_agent(llm, lc_tools, hub.pull("hwchase17/react"))
#   AgentExecutor(agent=agent, tools=lc_tools).invoke({"input": query})

# Offline yo'li: rule-based router (m15 ichida) -- har so'rov uchun tool tanlaydi
for q in ["Bu hujjatni qisqacha bayon qil",
          "Mehnat shartnomasi haqida nima deydi?",
          "Bu sharh ijobiymi?"]:
    print(q, "->", agent.route(q)["tool"])

### Tekshiring
1. "qisqacha bayon" qaysi vositaga olib bordi? (`summarize_text`)
2. "ijobiymi" qaysi vositaga? (`sentiment_classify`)
3. Hech qaysi kalit so'z mos kelmasa, agent qaysi vositani tanlaydi? (zaxira: `retrieve_docs`)

### O'zgartiring
`agent.route("...")` ga o'z so'rovingizni bering va qaysi vosita tanlanishini ko'ring.

### ✅ Checkpoint A
Agentni qaytadan quradi — oldingi kataklar ishlamagan bo'lsa, shu yerdan davom etamiz.

In [ ]:
if OFFLINE_FALLBACK or "agent" not in dir():
    agent = DocumentAssistantAgent(tools=make_tools())
print("Checkpoint A: agent tayyor, vositalar:", list(agent._tools.keys()))

## 4. Asosiy mavzu — so'nuvchi tayanch

Avval L15 [I1] dagi ReAct tool tanlashni tekshiramiz, so'ng vositalarni ulab agent quramiz, oxirida
murakkab so'rovni bajarib ReAct izini kuzatamiz.

### 4A. Namuna (men qilaman): ReAct to'g'ri vositani tanlaydi

L15 [I1]: "Uzum Market ilovasi yaxshimi?" hissiyot savoli $\to$ agent `sentiment_classify` ni tanlaydi.

In [ ]:
r = agent.route("Uzum Market ilovasi yaxshimi?")
print("Tanlangan vosita:", r["tool"], "| ishonch:", round(r["confidence"], 2))
assert r["tool"] == "sentiment_classify"   # Ma'ruza L15 [I1]-slayd bilan solishtiring
assert r["confidence"] > 0.7
print("To'g'ri! Agent sentiment_classify ni tanladi (ishonch > 0.7) -- L15 [I1] bilan mos.")

### 4B. Birgalikda (biz qilamiz): vositalarni agentga ulash

5 ta modulni tool dict ga ulab, agent yarating va marshrutlashni tekshiring.

In [ ]:
tools = make_tools()
my_agent = DocumentAssistantAgent(tools=tools)
print("Ulangan vositalar:", list(tools.keys()))

In [ ]:
assert set(tools.keys()) == {"sentiment_classify", "retrieve_docs",
                             "summarize_text", "spell_correct", "extract_entities"}
assert my_agent.route("Bu hujjatni qisqacha bayon qil")["tool"] == "summarize_text"
assert my_agent.route("Bu matndagi shahar nomlarini top")["tool"] == "extract_entities"
assert my_agent.route("imlo xatosini tuzat")["tool"] == "spell_correct"
print("To'g'ri! 5 vosita ulandi; marshrutlash kalit so'zlarga mos.")

### 4C. Birgalikda: agentni ishga tushirish (run)

Agentga so'rov bering — u vositani tanlaydi, chaqiradi va o'zbekcha javob qaytaradi.

In [ ]:
javob = my_agent.run("Mehnat shartnomasi haqida nima deydi?")
print(javob)
print("Iz:", my_agent.last_trace[0]["action"])

In [ ]:
assert isinstance(javob, str) and javob          # bo'sh bo'lmagan o'zbekcha javob
assert len(my_agent.last_trace) == 2               # Thought/Action + Observation
assert "action" in my_agent.last_trace[0]
assert "observation" in my_agent.last_trace[1]
print("To'g'ri! run -> str; last_trace Thought/Action/Observation saqladi.")

### 4D. Mustaqil (siz qilasiz): murakkab vazifa

Murakkab so'rov bering va ReAct izini to'liq kuzating. (Offline router 1 vosita tanlaydi; LLM agent ko'p
qadam qilardi — bu halol soddalashtirish.)

In [ ]:
murakkab = "Bu sharh ijobiymi yoki salbiy?"
natija = my_agent.run(murakkab)
print("Savol :", murakkab)
print("Javob :", natija)
print("Thought:", my_agent.last_trace[0]["thought"])
print("Action :", my_agent.last_trace[0]["action"])
print("Observation:", my_agent.last_trace[1]["observation"])

assert isinstance(natija, str) and natija
assert my_agent.last_trace[0]["action"] == "sentiment_classify"
print("To'g'ri! Murakkab so'rovda ham ReAct izi kuzatildi.")

## 5. Loyihaga ulash — `m15 DocumentAssistantAgent`

Bugungi kod `capstone/modules/m15_langchain_agent.py` ga yig'ilgan (interfeys `contracts.py` dan). Bu
**kapstone yakuni**: barcha modullaringizni bitta agentga birlashtiradi (himoya demo'sida ishlatiladi).

In [ ]:
api = DocumentAssistantAgent(tools=make_tools())
assert hasattr(api, "run")                       # shartnoma: run()
out = api.run("Bu mahsulot yaxshimi?")
print("Agent javobi:", out)
assert isinstance(out, str) and out
print("To'g'ri! m15 shartnomaga mos (run -> o'zbekcha str). Kapstone yakunlandi.")

**Git (o'z repozitoriyangizda):**
```bash
git add capstone/modules/m15_langchain_agent.py
git commit -m "P15: m15 DocumentAssistantAgent (kapstone yakuni)"
git push
```

## 6. Tadqiqot savoli + yakun

**Tajriba:** Turli so'rovlarga agent qaysi vositani tanlashini kuzating.

In [ ]:
for q in ["Bu fikr ijobiymi?", "Hujjatni qisqartir", "Qonun haqida ma'lumot ber",
          "Bu yerdagi joy nomlari qaysi?"]:
    print(q, "->", agent.route(q)["tool"])

**O'ylab ko'ring:** offline router kalit so'zga tayanadi — qaysi so'rovlarda u adashishi mumkin?
Haqiqiy LLM agent (LangChain) ko'p qadamli vazifani (sentiment $+$ qidiruv $+$ xulosa) qanday rejalashtiradi?
O'zbek tilida "Thought" yuritish ishonchlilikka qanday ta'sir qiladi (L15 [M])?

---
**Tabriklaymiz — kapstone "O'zbek hujjat yordamchisi" yakunlandi!** 16 kun davomida qurgan modullaringiz
bitta ishlaydigan agentga birlashdi.

**Chiqish chiptasi:** Butun kursda eng foydali bo'lgan modul/mavzu qaysi? (Bir jumla.)